# 02 — Project next IDX session with the TF15 model

Uses actual TF15 context through the newest completed candle. By default it
screens the 30 most liquid eligible stocks and ranks the predicted first
15-minute candle of the next weekday. CPU is supported; it is slower than CUDA.

Run notebook 01 first. If tomorrow is an IDX holiday, set `TARGET_DATE` manually.

In [1]:
from pathlib import Path
import importlib.util

HERE = Path.cwd().resolve()
if HERE.name != "Daily Screener":
    HERE = HERE / "Daily Screener"
module_path = HERE / "project_tf15_next_session.py"
assert module_path.exists(), f"Open this notebook from the ISTL repository: {HERE}"
spec = importlib.util.spec_from_file_location("tf15_projection", module_path)
tf15 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tf15)

## Configuration

In [2]:
CANDIDATE_COUNT = 30
LOOKBACK_BARS = 240
SAMPLE_PATHS = 3       # use 1 for a quick CPU smoke test; 5 for a stronger estimate
BATCH_SIZE = None      # automatic: CPU=2, CUDA=16
TARGET_DATE = None     # example: "2026-08-04"; None = next weekday after latest actual bar

In [ ]:
ranking, forecast_paths, metadata = tf15.run_projection(
    candidate_count=CANDIDATE_COUNT,
    lookback=LOOKBACK_BARS,
    paths=SAMPLE_PATHS,
    batch_size=BATCH_SIZE,
    target_date=TARGET_DATE,
)
metadata

{'device': 'cpu', 'context_end': '2026-08-03 14:45:00', 'target': '2026-08-04', 'candidates': 30}


config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

Loading weights from local directory


path 1/3:   0%|          | 0/15 [00:00<?, ?it/s]

In [ ]:
from IPython.display import display
styled = ranking.style.format({
    "anchor_close": "{:,.2f}", "expected_opening_bar_close": "{:,.2f}",
    "expected_return": "{:+.2%}", "median_return": "{:+.2%}",
    "probability_up": "{:.0%}", "downside_p10": "{:+.2%}",
}).background_gradient(subset=["expected_return"], cmap="RdYlGn")
display(styled)